# Original split: dataset, training and evaluation

Code for the **original split** of the honours thesis *Lightweight Object Detection for African Penguin Population Monitoring* (Hluma Xako, Rhodes University, 2026). It produced Experiments 1.1 and 4.1.

In this split, the African Penguin iNaturalist source keeps its own train, validation and test folders, and every other source is used for training only. Evaluation therefore uses African Penguin photographs only (855 test photographs).

| Section | Purpose |
|---|---|
| 1 | Build the merged dataset |
| 2 | Train YOLOv11n, YOLOv11m, YOLOv26n and YOLOv26m |
| 3 | Counting accuracy: MAE and photographs within 5% |
| 4 | OpenVINO export, test accuracy and latency |
| 5 | Native PyTorch CPU latency |
| 6 | Hyperparameter tuning of YOLOv11n |
| 7 | Visualising predictions and logging them to Weights & Biases |

Paths point to the machine the experiments ran on. Each training cell takes several hours on an RTX 4070.

## Setup

In [ ]:
# !pip install ultralytics openvino torchvision pandas matplotlib seaborn wandb

import os, glob, time, csv
import torch
from ultralytics import YOLO

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

dataset_dir = "/home/hluma/penguin_dataset_final"
data_yaml = dataset_dir + "/data.yaml"
results_dir = "/home/hluma/hluma_results"
os.makedirs(results_dir, exist_ok=True)

device = 0 if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Dataset:", data_yaml)

## 1. Build the merged dataset

Merges the African Penguin source (keeping its own splits), the Humboldt, Magellanic and Galápagos iNaturalist sources and the Roboflow sources (training only) into one single-class YOLO dataset. Duplicates are skipped using an MD5 hash of each file's first 8 KB, so only byte-identical copies are removed. The LOCKb camera is excluded.

In [ ]:
import shutil
import hashlib
from pathlib import Path


# PATHS

BASE = Path("/mnt/LinuxBackup2/Datasets_of_interest/Marine/Penguin")

OUT = Path("/home/hluma/penguin_dataset_final")

AFRICAN = BASE / "african-penguin-inaturalist-sam-veryclean"

INAT_EXTRA = [
    BASE / "humboldt-inaturalist-sam",
    BASE / "magellanicus-naturalist-sam",
    BASE / "galapagos-inaturalist-sam",
]

ROBOFLOW_TRAIN_SOURCES = [
    BASE / "RoboflowDatasets/SEGMENT/penguin-seg-combined",
    BASE / "RoboflowDatasets/SEGMENT/KIT315-sam",
    BASE / "RoboflowDatasets/SEGMENT/marine-and-aerial-segmentation-martin-sam",
    BASE / "RoboflowDatasets/SEGMENT/penguin-finder-marine-aerial-sam",
    BASE / "RoboflowDatasets/SEGMENT/MyFirstProject2-sam",
    BASE / "RoboflowDatasets/SEGMENT/marine-and-aerial-segmentation-sam",
    BASE / "RoboflowDatasets/SEGMENT/penguin-instance-sam",
    BASE / "RoboflowDatasets/SEGMENT/penguin-seg-sam",
    BASE / "RoboflowDatasets/SEGMENT/penguin-watch/geora-sam",
    BASE / "RoboflowDatasets/det/penguin_3single-sam",
    BASE / "RoboflowDatasets/det/penguin-water-sam",
    BASE / "RoboflowDatasets/det/penguin-water-hard-fp",
    BASE / "RoboflowDatasets/det/OCAT-sam",
    BASE / "RoboflowDatasets/det/pingwiny-spheniscus",
    BASE / "RoboflowDatasets/det/pingwiny-spheniscus2",
    BASE / "RoboflowDatasets/SEGMENT/multiclass/PenguinSpecies-Single-sam",
    BASE / "RoboflowDatasets/SEGMENT/original_splits/Penguin-Seaworld-sam3",
]

def img_hash(path):
    # A quick fingerprint of a file, used to spot duplicate images.
    # MD5 turns the file's raw bytes into a short code. Only the first 8 KB
    # (8,192 bytes) are read, which is fast. Two files get the same code only if
    # their first 8 KB are byte-for-byte identical, so a resized or re-saved copy
    # of a photo is NOT caught. (The grouped split uses dHash instead, which
    # compares how images look, and catches those copies too.)
    h = hashlib.md5()

    with open(path, "rb") as f:          # "rb" = read the raw bytes of the file
        h.update(f.read(8192))

    return h.hexdigest()                 # the code as text, e.g. "9e107d9d372bb682..."


def collect_pairs(img_dir, lbl_dir):
    # List every image in a folder that has a matching label file.

    pairs = []                           # (image path, label path) for each usable image

    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)

    if not img_dir.exists():             # a source may have no valid or test folder
        return pairs                     # then there is nothing to collect

    for img in img_dir.iterdir():        # every file in the images folder

        # skip anything that is not a picture
        if img.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        # the label has the same name as the image, ending in .txt
        lbl = lbl_dir / f"{img.stem}.txt"

        if lbl.exists():                 # keep the image only if it has a label
            pairs.append((img, lbl))

    return pairs


def copy_pairs(pairs, split, seen_hashes):
    # Copy images and labels into one split of the new dataset, skipping duplicates.

    img_out = OUT / split / "images"
    lbl_out = OUT / split / "labels"

    # create the folders; exist_ok=True means no error if they already exist
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    copied = 0

    for img, lbl in pairs:

        h = img_hash(img)                # fingerprint of this image

        if h in seen_hashes:             # an identical file was already copied,
            continue                     # so skip this one

        seen_hashes.add(h)

        # Copy using the ORIGINAL file names. If two different sources contain
        # different photos with the same name (e.g. "IMG_0001.jpg"), the second
        # copy overwrites the first. This is why the grouped split gives every
        # file a new, unique name. copy2 also keeps each file's date stamps.
        shutil.copy2(img, img_out / img.name)
        shutil.copy2(lbl, lbl_out / lbl.name)

        copied += 1

    return copied

# START PROCESS. 
print("=" * 60)
print("Building Penguin Dataset")
print("=" * 60)

# One set of fingerprints shared by every step, so a photo that appears in two
# sources is only copied the first time it is met.
seen = set()

# STEP 1: 
# African Penguin dataset: keeps its own train, valid and test folders.
# It goes first, so when a photo appears in several sources, the African
# Penguin copy is the one kept. This makes validation and test African
# Penguin photos only.


print("\nStep 1: African Penguin dataset")

train_pairs = collect_pairs(
    AFRICAN / "train" / "images",
    AFRICAN / "train" / "labels"
)

valid_pairs = collect_pairs(
    AFRICAN / "valid" / "images",
    AFRICAN / "valid" / "labels"
)

test_pairs = collect_pairs(
    AFRICAN / "test" / "images",
    AFRICAN / "test" / "labels"
)

# train is copied first, then valid, then test. An identical copy of a training
# photo in the test folder is therefore skipped, which keeps exact copies out of
# the test set (the source's test folder... 855 were kept).
train_count = copy_pairs(
    train_pairs,
    "train",
    seen
)

valid_count = copy_pairs(
    valid_pairs,
    "valid",
    seen
)

test_count = copy_pairs(
    test_pairs,
    "test",
    seen
)

print(
    f"African Penguin -> "
    f"train:{train_count} "
    f"valid:{valid_count} "
    f"test:{test_count}"
)

# STEP 2
# Other iNaturalist species -> TRAIN ONLY
# All three of their folders (train, valid, test) go into training,
# because evaluation uses African Penguins only.


print("\nStep 2: Other iNaturalist species")

for src in INAT_EXTRA:

    if not src.exists():
        print(f"SKIP: {src.name}")
        continue

    total_added = 0

    for split in ["train", "valid", "test"]:

        pairs = collect_pairs(
            src / split / "images",
            src / split / "labels"
        )

        total_added += copy_pairs(
            pairs,
            "train",                     # always into training
            seen
        )

    print(
        f"{src.name}: "
        f"{total_added} images added"
    )


# STEP 3
# Roboflow datasets -> TRAIN ONLY
# Only each dataset's own "train" folder is used. Its valid and test
# folders are left out.


print("\nStep 3: Roboflow datasets")

for src in ROBOFLOW_TRAIN_SOURCES:

    if not src.exists():
        print(f"SKIP: {src.name}")
        continue

    pairs = collect_pairs(
        src / "train" / "images",
        src / "train" / "labels"
    )

    copied = copy_pairs(
        pairs,
        "train",
        seen
    )

    print(
        f"{src.name}: "
        f"{copied} images added"
    )

# DATA YAML
# Tells Ultralytics where the images are, and that there is one class.


yaml_content = f"""
train: {OUT}/train/images
val: {OUT}/valid/images
test: {OUT}/test/images

nc: 1
names: ['penguin']
"""

with open(OUT / "data.yaml", "w") as f:
    f.write(yaml_content.strip())        # strip() removes the blank first and last lines


# Counts the files actually in each folder. Because overwritten files are
# counted only once, this can be lower than the number "copied" above.
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

for split in ["train", "valid", "test"]:

    count = len(
        list(
            (OUT / split / "images").glob("*")
        )
    )

    print(f"{split:6s}: {count}")        # :6s pads the name to 6 characters, to line up

print()
print(f"data.yaml -> {OUT}/data.yaml")

### Dataset check

In [ ]:
import os

print("Train images:", len(os.listdir(dataset_dir + "/train/images")))
print("Train labels:", len(os.listdir(dataset_dir + "/train/labels")))


print("Valid images:", len(os.listdir(dataset_dir + "/valid/images")))
print("Valid labels:", len(os.listdir(dataset_dir + "/valid/labels")))


print("Test images:", len(os.listdir(dataset_dir + "/test/images")))
print("Test labels:", len(os.listdir(dataset_dir + "/test/labels")))

## 2. Training

Each model starts from COCO-pretrained Ultralytics weights and trains for up to 100 epochs at 640 pixels, stopping early after 20 epochs without improvement. The optimiser is left at the Ultralytics default (`auto`).

### YOLOv11n
The run used in the thesis is `runs/yolov11n-3`.

In [ ]:
#train yolo11 nano 
yolo11n_model = YOLO('yolo11n.pt')

yolo11n_results = yolo11n_model.train(
    data      = data_yaml,
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,
    workers   = 8,
    device    = device,
    project   = os.path.join(results_dir, 'runs'),
    name      = 'yolov11n',
    patience  = 20,
    save      = True,
    val       = True,
    plots     = True,
)

print("\n--- YOLOv11n: test set evaluation ---")
yolo11n_model = YOLO(os.path.join(results_dir, 'runs/yolov11n-3/weights/best.pt'))
yolo11n_test = yolo11n_model.val(
    data    = data_yaml,
    split   = 'test',
    device  = device,
    verbose = True,
)
print(f"YOLOv11n TEST mAP@50: {yolo11n_test.box.map50:.4f}")

### YOLOv11m
This run used a batch size of 8.

In [ ]:
yolo11m_model = YOLO('yolo11m.pt')

yolo11_results = yolo11m_model.train(
    data      = data_yaml, 
    epochs    = 100, 
    imgsz     = 640, 
    batch     = 8,
    workers   = 2,
    device    = device,  
    project   = os.path.join(results_dir, 'runs'), 
    name      = 'yolov11m', 
    patience  = 20,     #stop early if needed
    cache     = False,
    save      = True, 
    val       = True, 
    plots     = True, 
)

print('YOLOv11m training complete.')

In [ ]:
print("YOLOv11m: Test set evaluation:")
yolo11m_best= YOLO(os.path.join(results_dir, 'runs/yolov11m/weights/best.pt'))
yolo11m_test = yolo11m_best.val(
    data    = data_yaml, 
    split   = 'test', 
    device  = device, 
    verbose = True,
)
print(f"YOLOv11m TEST mAP@50: {yolo11m_test.box.map50:.4f}")

### YOLOv26m
The run used in the thesis is `runs/yolo26m-2`.

In [ ]:
# Falls back to the nano model if the medium weights are not available
try:
    yolo26_model = YOLO('yolo26m.pt')
    yolo26_size = 'm'
except Exception:
    print('yolo26m.pt not available, using yolo26n.pt') 
    yolo26_model = YOLO('yolo26n.pt')
    yolo26_size = 'n'

yolo26_results = yolo26_model.train(
    data       = data_yaml,
    epochs     = 100,
    imgsz      = 640,
    batch      = 16,
    device     = device,
    project    = os.path.join(results_dir, 'runs'),
    name       = f'yolo26{yolo26_size}',
    patience   = 20,
    save       = True,
    val        = True,
    plots      = True,
)
print('YOLOv26m training complete.')

### YOLOv26n

In [ ]:
yolov26n_model = YOLO('yolo26n.pt')


yolo26n_results = yolov26n_model.train(
    data      = data_yaml,
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,
    workers   = 8,
    device    = device,
    project   = os.path.join(results_dir, 'runs'),
    name      = 'yolov26n',
    patience  = 20,
    save      = True,
    val       = True,
    plots     = True,
)

print('YOLOv26n training complete.')

results = '/home/hluma/hluma_results'
data_yaml = '/home/hluma/penguin_dataset_final/data.yaml'

yolov26n_model = YOLO(os.path.join(results_dir, "runs/yolov26n/weights/best.pt"))

print("\n--- YOLOv26n: test set evaluation ---")
yolov26n_test = yolov26n_model.val(
    data    = data_yaml,
    split   = 'test',
    device  = device,
    verbose = True,
)
print(f"YOLOv26n TEST mAP@50: {yolov26n_test.box.map50:.4f}")
print(f"YOLOv26n TEST mAP@50-95: {yolov26n_test.box.map:.4f}")
print(f"YOLOv26n TEST Precision: {yolov26n_test.box.p.mean():.4f}")
print(f"YOLOv26n TEST Recall: {yolov26n_test.box.r.mean():.4f}")

## 3. Counting accuracy

For each test photograph with penguins, the true count is the number of label lines and the predicted count is the number of boxes above a confidence of 0.25. MAE is the mean absolute difference, and *within 5%* is the share of photographs whose count is off by 5% or less. Photographs without penguins are skipped. These results were printed rather than saved to a file.

In [ ]:
print("\nPopulation Counting (MAE)")
print("-" * 40)
import os
import glob
from ultralytics import YOLO

#the paths to the images and their labels
TEST_IMG = os.path.join(dataset_dir, "test/images")
TEST_LBL = os.path.join(dataset_dir, "test/labels")

MODELS = {
    "YOLOv11n": os.path.join(results_dir, "runs/yolov11n-3/weights/best.pt"),
    "YOLOv11m": os.path.join(results_dir, "runs/yolov11m/weights/best.pt"),
    "YOLOv26m": os.path.join(results_dir, "runs/yolo26m-2/weights/best.pt"),
    "YOLOv26n": os.path.join(results_dir, "runs/yolov26n/weights/best.pt"),
}

#get ground truth counts from label files. 
def get_ground_truth_counts(label_folder):
    counts = {} #counts is a dictionary

    #loop through label files
    for label_file in glob.glob(label_folder + "/*.txt"): #since label files have .txt at the end
        img_name = os.path.splitext(os.path.basename(label_file))[0]

        with open(label_file) as f:
            num_penguins = len(f.readlines())
        
        counts[img_name] = num_penguins
    return counts
#Read the true counts
ground_truth = get_ground_truth_counts(TEST_LBL)

# Find all test images
test_images = glob.glob(TEST_IMG + "/*.jpg")
test_images += glob.glob(TEST_IMG + "/*.png")


for model_name, weights_path in MODELS.items():

    print(f"\nEvaluating {model_name}...")

    # Skip if weights don't exist
    if not os.path.exists(weights_path):
        print("Weights not found.")
        continue

    model = YOLO(weights_path)

    errors = []
    within_5_percent = 0
    total_images = 0

    for image_path in test_images:

        image_name = os.path.splitext(os.path.basename(image_path))[0]

        # True number of penguins
        true_count = ground_truth.get(image_name, 0)

        # Ignore images with no penguins
        if true_count == 0:
            continue

        # Model prediction
        result = model(image_path, conf=0.25, verbose=False)[0]

        # Number of predicted penguins
        predicted_count = len(result.boxes)

        # Counting error
        error = abs(predicted_count - true_count)

        errors.append(error)

        # Was it within 5%?
        percent_error = (error / true_count) * 100

        if percent_error <= 5:
            within_5_percent += 1

        total_images += 1

    # Final statistics
    mae = sum(errors) / len(errors)

    percentage_within_5 = (
        within_5_percent / total_images
    ) * 100

    print(f"MAE: {mae:.2f} penguins")
    print(f"Within 5% of true count: {percentage_within_5:.1f}%")
    print(f"Images analysed: {total_images}")

## 4. OpenVINO export, accuracy and latency

Each model is exported to OpenVINO at FP32 with a fixed 640 x 640 input and evaluated on the test set on the CPU. Latency is timed on every test photograph after 10 untimed warm-up inferences. The code also requests OpenVINO's GPU device (`intel:gpu.0`). At the time, the Intel GPU compute runtime was not installed on this machine, so OpenVINO could not reach the integrated GPU, and the `intel:gpu.0` timings most likely ran on the CPU. They are not used in the thesis, which reports CPU results only. Results are written to `openvino_benchmark_results.csv`.

In [ ]:
print("\nOpenVINO Export and Benchmarking")
print("-" * 40)

import os
import time
import csv
from ultralytics import YOLO


OPENVINO_DEVICES = ["intel:cpu", "intel:gpu.0"] 

# warm-up frames are not timed
WARMUP_FRAMES = 10

#create a csv file to store the openvino results 
results_csv_path = os.path.join(results_dir, "openvino_benchmark_results.csv")
all_results = []

for model_name, weights_path in MODELS.items():
    print(f"\nProcessing {model_name}...")

    if not os.path.exists(weights_path):
        print("Weights not found.")
        continue

    # Export to OpenVINO (only needs to happen once per model)
    model = YOLO(weights_path)

    export_path = model.export(
        format="openvino",
        imgsz=640,
        dynamic=False,
        half=False,
    )
    print(f"{model_name} exported successfully.")
    print(f"OpenVINO model saved to: {export_path}")

    # Accuracy on the test set, on the CPU 
    ov_model_for_val = YOLO(export_path)
    metrics = ov_model_for_val.val(
        data=data_yaml,
        split="test",
        imgsz=640,
        device="intel:cpu",
        verbose=False,
    )
    map50 = metrics.box.map50
    map5095 = metrics.box.map
    print(f"{model_name} OpenVINO Test mAP@50: {map50:.4f}")
    print(f"{model_name} OpenVINO Test mAP@50-95: {map5095:.4f}")

    # Latency benchmarking 
    for ov_device in OPENVINO_DEVICES:

        print(f"\n  Benchmarking {model_name} on OpenVINO device: {ov_device}")

        try:
            ov_model = YOLO(export_path)

            # Warm-up (excluded from timing)
            for image_path in test_images[:WARMUP_FRAMES]:
                ov_model(
                    image_path,
                    imgsz=640,
                    device=ov_device,
                    verbose=False,
                )

            # Timed pass 
            per_image_times = []
            for image_path in test_images:
                start = time.perf_counter()
                ov_model(
                    image_path,
                    imgsz=640,
                    device=ov_device,
                    verbose=False,
                )
                per_image_times.append(time.perf_counter() - start)

            avg_latency_ms = (sum(per_image_times) / len(per_image_times)) * 1000
            std_latency_ms = (
                (sum((t * 1000 - avg_latency_ms) ** 2 for t in per_image_times)
                 / len(per_image_times)) ** 0.5
            )
            fps = 1000 / avg_latency_ms

            print(f"  {model_name} [{ov_device}] Average latency: "
                  f"{avg_latency_ms:.2f} ms/image (+/- {std_latency_ms:.2f} ms)")
            print(f"  {model_name} [{ov_device}] FPS: {fps:.2f}")

            all_results.append({
                "model": model_name,
                "device": ov_device,
                "map50": round(map50, 4),
                "map50_95": round(map5095, 4),
                "avg_latency_ms": round(avg_latency_ms, 2),
                "std_latency_ms": round(std_latency_ms, 2),
                "fps": round(fps, 2),
                "n_images": len(per_image_times),
                "warmup_frames": WARMUP_FRAMES,
                "precision": "FP32",
                "error": "",
            })

        except Exception as e:
            print(f"  Could not benchmark {model_name} on {ov_device}: {e}")
            all_results.append({
                "model": model_name,
                "device": ov_device,
                "map50": round(map50, 4),
                "map50_95": round(map5095, 4),
                "avg_latency_ms": None,
                "std_latency_ms": None,
                "fps": None,
                "n_images": 0,
                "warmup_frames": WARMUP_FRAMES,
                "precision": "FP32",
                "error": str(e),
            })

# Write all results to a single CSV 
if all_results:
    fieldnames = [
        "model", "device", "map50", "map50_95",
        "avg_latency_ms", "std_latency_ms", "fps",
        "n_images", "warmup_frames", "precision", "error",
    ]
    with open(results_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in all_results:
            writer.writerow(row)

    print(f"\nAll OpenVINO benchmark results written to: {results_csv_path}")
else:
    print("\nNo results were collected - check that MODELS paths are correct.")

## 5. Native PyTorch CPU latency

The same models, run directly in PyTorch on the CPU without exporting, as the baseline for OpenVINO. Results are written to `pytorch_cpu_benchmark_results.csv`.

In [ ]:
# Native PyTorch CPU benchmark (original split).
# Times the trained models run directly in PyTorch on the CPU, WITHOUT exporting
# them to OpenVINO. This is the "before optimisation" baseline that OpenVINO is
# compared against.
# Uses test_images (from the counting cell) and WARMUP_FRAMES (from the OpenVINO
# cell), so those cells must be run first.

print("\nNative PyTorch CPU Benchmarking")
print("-" * 40)

pytorch_results = []

for model_name, weights_path in MODELS.items():
    if not os.path.exists(weights_path):          # skip a model that has not been trained
        print(f"{model_name}: weights not found, skipping.")
        continue

    print(f"\nBenchmarking {model_name} on native PyTorch CPU...")
    model = YOLO(weights_path)  # the .pt file, not exported

    # Warm-up: the first predictions are slow while the model loads into memory,
    # so the first 10 are run but not timed
    for image_path in test_images[:WARMUP_FRAMES]:
        model(image_path, imgsz=640, device="cpu", verbose=False)   # device="cpu" forces the CPU

    # Timed pass: one prediction per test photo, each timed with a stopwatch
    per_image_times = []
    for image_path in test_images:
        start = time.perf_counter()
        model(image_path, imgsz=640, device="cpu", verbose=False)
        per_image_times.append(time.perf_counter() - start)        # time in seconds

    # Average time per photo, converted from seconds to milliseconds
    avg_latency_ms = (sum(per_image_times) / len(per_image_times)) * 1000
    fps = 1000 / avg_latency_ms                    # photos per second

    print(f"  {model_name} [pytorch:cpu] Average latency: {avg_latency_ms:.2f} ms/image")
    print(f"  {model_name} [pytorch:cpu] FPS: {fps:.2f}")

    pytorch_results.append({
        "model": model_name,
        "device": "pytorch:cpu",
        "avg_latency_ms": round(avg_latency_ms, 2),
        "fps": round(fps, 2),
        "n_images": len(per_image_times),
        "warmup_frames": WARMUP_FRAMES,
    })

# Save to pytorch_cpu_benchmark_results.csv
import csv
with open(os.path.join(results_dir, "pytorch_cpu_benchmark_results.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["model", "device", "avg_latency_ms", "fps", "n_images", "warmup_frames"])
    writer.writeheader()
    for row in pytorch_results:
        writer.writerow(row)

print("\nDone. Results saved.")

## 6. Hyperparameter tuning (Experiment 4.1)

In [ ]:
# Hyperparameter search for YOLOv11n (Experiment 4.1): 10 Ray Tune trials with Optuna,
# 20 epochs each, continuing from the trained baseline. Needs yolo_any_tune.py and
# bench_schema.py in the same folder (see ../tools). Both scripts were written by Prof. Dane Brown.
!python3 yolo_any_tune.py \
    --data /home/hluma/penguin_dataset_final \
    --model /home/hluma/hluma_results/runs/yolov11n-3/weights/best.pt \
    --task detect --tune --tune_iter 10 --epochs 20

## 7. Visualising predictions

Side-by-side predictions of two models on one test photograph, then the same photograph at four confidence thresholds.

In [ ]:
MODELS_TO_SHOW = {
    "YOLO11m": "/home/hluma/hluma_results/runs/yolov11m/weights/best.pt",
    "YOLO26m": "/home/hluma/hluma_results/runs/yolo26m-2/weights/best.pt",
}
image_to_visualise = "/home/hluma/penguin_dataset_final/test/images/68894828_111665916.jpg"
output_dir = "/home/hluma/hluma_results/presentation_results"
os.makedirs(output_dir, exist_ok=True)

for name, weights in MODELS_TO_SHOW.items():
    if not os.path.exists(weights):
        print(f"Weights not found: {weights}")
        continue
    YOLO(weights).predict(
        source=image_to_visualise,
        conf=0.25,
        imgsz=640,
        save=True,
        project=output_dir,
        name=name,
        show_labels=False,
        show_conf=False,
        exist_ok=True,
    )
    print(f"{name} prediction saved.")
print(f"Results saved to: {output_dir}")

In [ ]:
image = "/home/hluma/penguin_dataset_final/test/images/139705596_239003017.jpg"
thresholds = [0.10, 0.25, 0.50, 0.80]
model = YOLO("/home/hluma/hluma_results/runs/yolo26m-2/weights/best.pt")

for conf in thresholds:
    model.predict(
        source=image,
        conf=conf,
        save=True,
        project="confidence_demo",
        name=f"conf_{conf}",
        show_conf=True,
        show_labels=True,
        exist_ok=True,
    )

### Logging predictions to Weights & Biases

Logs each test photograph with its predicted boxes, latency and number of detections, for browsing in the W&B dashboard.

In [ ]:
# Log predictions to Weights & Biases (W&B), an online dashboard for ML experiments.
# Each test photo is uploaded with the model's boxes drawn on it, together with
# its prediction time and number of detections, so results can be browsed visually.

import wandb
from pathlib import Path

MODEL_PATH = "/home/hluma/hluma_results/runs/yolov11m/weights/best.pt"   # model to show
TEST_IMAGES = "/home/hluma/penguin_dataset_final/test/images"            # photos to run it on
CONF_THRESH = 0.25                                                       # same threshold as counting

# Start a new W&B "run": one entry on the dashboard.
#  project: the dashboard folder this run belongs to
#  name:    a label for this run, e.g. "best-inference"
#  config:  the settings, saved alongside the results for reference
wandb.init(
    project="penguin-detection",
    name=f"{Path(MODEL_PATH).stem}-inference",
    config={"model": MODEL_PATH, "conf_thresh": CONF_THRESH},
)

model = YOLO(MODEL_PATH)
# every .jpg and .png in the test folder
all_images = list(Path(TEST_IMAGES).glob("*.jpg")) + list(Path(TEST_IMAGES).glob("*.png"))
failed_images = []   # photos that could not be processed

for im_path in all_images:
    try:
        # time one prediction, from start to finish, in milliseconds
        t0 = time.perf_counter()
        results = model.predict(source=str(im_path), conf=CONF_THRESH, verbose=False)
        latency_ms = (time.perf_counter() - t0) * 1000

        r = results[0]   # the result for this one photo

        # Convert the model's boxes into the format W&B needs to draw them.
        # b.xyxy holds each box's corners: left (x1), top (y1), right (x2), bottom (y2).
        wandb_boxes = {
            "predictions": {
                "box_data": [
                    {
                        "position": {
                            "minX": float(b.xyxy[0][0]), "minY": float(b.xyxy[0][1]),
                            "maxX": float(b.xyxy[0][2]), "maxY": float(b.xyxy[0][3]),
                        },
                        "class_id": int(b.cls[0]),                          # 0 = penguin
                        "box_caption": f"penguin {float(b.conf[0]):.2f}",   # label shown on the box
                        "scores": {"confidence": float(b.conf[0])},         # lets W&B filter by confidence
                    }
                    for b in r.boxes          # one entry per predicted box
                ],
                "class_labels": {0: "penguin"},   # the name for class 0
            }
        }

        # Upload this photo with its boxes, and record its time and box count
        wandb.log({
            "predictions": wandb.Image(str(im_path), boxes=wandb_boxes),
            "latency_ms": latency_ms,
            "num_detections": len(r.boxes),
        })

    except Exception as e:
        # A damaged photo is skipped and noted, instead of stopping the whole run
        print(f"Skipping unreadable image: {im_path} ({e})")
        failed_images.append(str(im_path))

print(f"{len(all_images) - len(failed_images)} images logged, {len(failed_images)} skipped.")
wandb.finish()   # close the run, so the dashboard marks it as complete